In [ ]:
!pip install pyspark

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 317.3/317.3 MB 2.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for pyspark: filename=pyspark-3.5.2-py2.py3-none-any.whl size=317812365 sha256=6f79afccd387769d60967508ac309c0f20dcf9728466121eccc37fad3d98b6eb
  Stored in directory: /root/.cache/pip/wheels/34/34/bd/03944534c44b677cd5859f248090daa9fb27b3c8f8e5f49574
Successfully built pyspark


In [ ]:
import pandas as pd
import random

# Generate sample data
def generate_sample_data(num_rows=1000):
    data = {
        'id': range(1, num_rows + 1),
        'name': [f'Name_{i}' for i in range(1, num_rows + 1)],
        'age': [random.randint(18, 60) for _ in range(num_rows)],
        'salary': [random.randint(30000, 120000) for _ in range(num_rows)],
    }
    return pd.DataFrame(data)

sample_data = generate_sample_data(1000)

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, avg, count

In [ ]:
spark = SparkSession.builder.appName("DataFrame and Dataset API").getOrCreate()

In [ ]:
# Convert to Spark DataFrame
df = spark.createDataFrame(sample_data)

In [ ]:
df.show()

+---+-------+---+------+
| id|   name|age|salary|
+---+-------+---+------+
|  1| Name_1| 45| 36993|
|  2| Name_2| 30| 91493|
|  3| Name_3| 32| 59727|
|  4| Name_4| 47|104428|
|  5| Name_5| 58| 31508|
|  6| Name_6| 25| 46504|
|  7| Name_7| 55| 72718|
|  8| Name_8| 25| 97872|
|  9| Name_9| 20| 80751|
| 10|Name_10| 24| 89440|
| 11|Name_11| 34| 48676|
| 12|Name_12| 38|114675|
| 13|Name_13| 27| 52997|
| 14|Name_14| 18| 32439|
| 15|Name_15| 45| 67164|
| 16|Name_16| 32| 55645|
| 17|Name_17| 51|112342|
| 18|Name_18| 37| 73848|
| 19|Name_19| 24| 97655|
| 20|Name_20| 45| 53652|
+---+-------+---+------+
only showing top 20 rows



In [ ]:
filtered_data = df.filter(col("age")>30).select("id", "name", "age")
print("people over 30")
filtered_data.show()

people over 30
+---+-------+---+
| id|   name|age|
+---+-------+---+
|  1| Name_1| 45|
|  3| Name_3| 32|
|  4| Name_4| 47|
|  5| Name_5| 58|
|  7| Name_7| 55|
| 11|Name_11| 34|
| 12|Name_12| 38|
| 15|Name_15| 45|
| 16|Name_16| 32|
| 17|Name_17| 51|
| 18|Name_18| 37|
| 20|Name_20| 45|
| 21|Name_21| 36|
| 23|Name_23| 44|
| 26|Name_26| 50|
| 27|Name_27| 31|
| 28|Name_28| 35|
| 30|Name_30| 50|
| 31|Name_31| 60|
| 32|Name_32| 35|
+---+-------+---+
only showing top 20 rows



In [ ]:
# Aggregations: Calculate average salary by age group
avg_salary_by_age = df.groupBy("age").agg(avg("salary").alias("average_salary"))
print("Average salary by age:")
avg_salary_by_age.show()

Average salary by age:
+---+------------------+
|age|    average_salary|
+---+------------------+
| 26| 72930.69230769231|
| 29| 73423.38461538461|
| 19|          77819.52|
| 54| 68704.83333333333|
| 22| 73433.63157894737|
| 34| 68286.13636363637|
| 50| 71480.64285714286|
| 57| 71148.42857142857|
| 32|           72740.0|
| 43| 76491.95238095238|
| 31| 73871.54545454546|
| 39| 70292.66666666667|
| 25| 73536.36363636363|
| 58| 66943.31034482758|
| 27| 65033.19047619047|
| 51|64459.379310344826|
| 56|         80926.375|
| 52| 80566.63636363637|
| 41| 71580.56521739131|
| 33| 70997.29166666667|
+---+------------------+
only showing top 20 rows



In [ ]:
total_count = df.count()
print(f"Total number of records: {total_count}")

Total number of records: 1000


In [ ]:
df.cache()

DataFrame[id: bigint, name: string, age: bigint, salary: bigint]

In [ ]:
# Perform another action to demonstrate caching
salary_count = df.groupBy("salary").agg(count("id").alias("count")).orderBy("count", ascending=False)
print("Salary count (cached):")
salary_count.show(10)

Salary count (cached):
+------+-----+
|salary|count|
+------+-----+
| 51115|    2|
| 99143|    2|
|117095|    2|
| 82458|    2|
| 59429|    2|
|112568|    1|
| 76901|    1|
| 49967|    1|
|117383|    1|
| 84548|    1|
+------+-----+
only showing top 10 rows



In [ ]:
spark.stop()

In [ ]:
spark = SparkSession.builder \
    .appName("Memory Configuration Example") \
    .config("spark.driver.memory", "2g") \  # Set driver memory to 2 GB
    .config("spark.executor.memory", "4g") \  # Set executor memory to 4 GB
    .config("spark.executor.memoryOverhead", "512m") \  # Set executor memory overhead to 512 MB
    .config("spark.memory.fraction", "0.75") \  # 75% of memory for execution and storage
    .config("spark.memory.storageFraction", "0.5") \  # 50% of storage memory for caching
    .getOrCreate()